# 15 — ERP Sanity Checks Across Datasets

In [ ]:
# Run from the repository root so that relative result paths resolve
# identically regardless of where this notebook lives.
from oddball_eeg.paths import use_project_root

PROJECT_ROOT = use_project_root()

In [ ]:
from __future__ import annotations

from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

from oddball_eeg.bids import make_epoching_config_from_experiment
from oddball_eeg.config import get_experiment
from oddball_eeg.io import save_dataframe
from oddball_eeg.loader import load_subjects_grouped
from oddball_eeg.preprocessing import apply_crop_to_loaded_dataset
from oddball_eeg.visualization import (
    round_numeric_columns,
    save_figure,
    set_pandas_display_options,
    valid_topomap_picks,
)


set_pandas_display_options()

EXPERIMENT_NAMES = [
    "ds003061_within_subject_max30_wide_timebin",
    "ds005863_within_subject_max30_wide_timebin",
    "ds006466_within_subject_max30_wide_timebin",
    "ds006480_within_subject_max30_wide_timebin",
]

ROI_BY_DATASET_LABEL = {
    "ds003061": ["Pz", "P3", "P4", "CPz", "Cz", "CP1", "CP2"],
    "ds005863": ["Pz", "P3", "P4", "CP1", "CP2"],
    "ds006466": ["E62", "E65", "E55", "E60", "E61", "E54", "E53"],
    "ds006480_pooled": ["E62", "E65", "E55", "E60", "E61", "E54", "E53"],
}

P300_WINDOW = (0.25, 0.50)
ROI_CANDIDATES = ["Pz", "P3", "P4", "CPz", "Cz", "CP1", "CP2"]

OUT_ROOT = Path("results") / "thesis_artifacts" / "erp_sanity"
FIG_DIR = OUT_ROOT / "figures"
TABLE_DIR = OUT_ROOT / "tables"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Thesis-ready plotting style
# ------------------------------------------------------------

plt.style.use("default")

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "text.color": "black",
    "axes.labelcolor": "black",
    "axes.edgecolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
})

# Stores clean waveform data for final dissertation figures.
erp_waveform_rows = []

In [ ]:
def pick_roi_indices(ch_names, candidates):
    name_to_idx = {ch.upper(): i for i, ch in enumerate(ch_names)}
    idx = [name_to_idx[ch.upper()] for ch in candidates if ch.upper() in name_to_idx]
    if idx:
        return idx
    for fb in ["PZ", "CZ"]:
        if fb in name_to_idx:
            return [name_to_idx[fb]]
    return [0]


def _set_montage_robust(info, montage_name: str):
    """
    Try setting a montage while remaining compatible with different MNE versions.
    """
    info = info.copy()
    montage = mne.channels.make_standard_montage(montage_name)

    try:
        info.set_montage(
            montage,
            on_missing="ignore",
            match_case=False,
            match_alias=True,
        )
    except TypeError:
        info.set_montage(
            montage,
            on_missing="ignore",
        )

    return info


def make_eeg_info(ch_names, sfreq):
    """
    Build an MNE Info object and automatically choose the montage that gives
    the largest number of valid electrode positions.

    This supports:
    - standard 10-20 channel names such as Fz, Cz, Pz
    - EGI/HydroCel names such as E1, E2, ..., E65
    """
    ch_names = [str(ch) for ch in ch_names]
    info_base = mne.create_info(
        ch_names=ch_names,
        sfreq=float(sfreq),
        ch_types="eeg",
    )

    has_egi_names = any(re.fullmatch(r"E\d+", ch.upper()) for ch in ch_names)

    if has_egi_names:
        montage_candidates = [
            "GSN-HydroCel-65_1.0",
            "GSN-HydroCel-64_1.0",
            "GSN-HydroCel-128",
            "GSN-HydroCel-129",
            "standard_1020",
            "standard_1005",
            "biosemi64",
            "biosemi128",
        ]
    else:
        montage_candidates = [
            "standard_1020",
            "standard_1005",
            "biosemi64",
            "biosemi128",
            "GSN-HydroCel-65_1.0",
            "GSN-HydroCel-64_1.0",
            "GSN-HydroCel-128",
            "GSN-HydroCel-129",
        ]

    best_info = info_base.copy()
    best_montage = None
    best_n_valid = 0

    for montage_name in montage_candidates:
        try:
            info_try = _set_montage_robust(info_base, montage_name)
            n_valid = len(valid_topomap_picks(info_try))

            if n_valid > best_n_valid:
                best_info = info_try
                best_montage = montage_name
                best_n_valid = n_valid

        except Exception:
            continue

    print(
        f"[topomap] selected montage: {best_montage}, "
        f"valid channels: {best_n_valid}/{len(ch_names)}"
    )

    return best_info, best_montage, best_n_valid


def compute_erp_summary_for_dataset(cfg):
    epoching_cfg = make_epoching_config_from_experiment(cfg)

    data_full = load_subjects_grouped(
        dataset=cfg.dataset.dataset,
        task_name=cfg.dataset.task,
        subjects=cfg.subjects.subjects,
        epoching_cfg=epoching_cfg,
        chunk_size=cfg.splits.group_chunk_size,
        apply_baseline=True,
        verbose=True,
    )

    # Use full epoch for ERP sanity, not model crop.
    data = data_full

    times = data.times
    ch_names = data.ch_names
    sfreq = data.sfreq
    roi_idx = pick_roi_indices(ch_names, ROI_CANDIDATES)

    p300_mask = (times >= P300_WINDOW[0]) & (times <= P300_WINDOW[1])
    if p300_mask.sum() == 0:
        raise ValueError("P300 window has no samples.")

    subject_rows = []
    std_erps = []
    tgt_erps = []
    diff_erps = []

    for sub in data.subjects:
        X = data.X_by_sub[sub]
        y = data.y_by_sub[sub]

        std = X[y == 0].mean(axis=0)
        tgt = X[y == 1].mean(axis=0)
        diff = tgt - std

        std_erps.append(std)
        tgt_erps.append(tgt)
        diff_erps.append(diff)

        roi_diff = diff[roi_idx].mean(axis=0)
        p300_segment = roi_diff[p300_mask]
        p300_times = times[p300_mask]

        peak_local_idx = int(np.argmax(np.abs(p300_segment)))
        peak_time = float(p300_times[peak_local_idx])
        peak_amp = float(p300_segment[peak_local_idx])
        mean_amp = float(p300_segment.mean())

        subject_rows.append({
            "dataset": cfg.dataset.dataset,
            "task": cfg.dataset.task,
            "event_preset": cfg.dataset.event_preset,
            "subject": sub,
            "n_trials": int(len(y)),
            "n_standard": int((y == 0).sum()),
            "n_target": int((y == 1).sum()),
            "target_rate": float((y == 1).mean()),
            "roi_channels": ",".join([ch_names[i] for i in roi_idx]),
            "p300_mean_amp": mean_amp,
            "p300_peak_amp": peak_amp,
            "p300_peak_latency_s": peak_time,
        })

    std_grand = np.mean(std_erps, axis=0)
    tgt_grand = np.mean(tgt_erps, axis=0)
    diff_grand = np.mean(diff_erps, axis=0)

    return data, pd.DataFrame(subject_rows), std_grand, tgt_grand, diff_grand

In [ ]:
def _unit_normalize_positions(pos: np.ndarray) -> np.ndarray:
    """
    Normalize 3D positions to unit vectors.

    This makes nearest-channel matching more robust across montages whose
    coordinate scales may differ slightly.
    """
    pos = np.asarray(pos, dtype=float)
    norms = np.linalg.norm(pos, axis=1, keepdims=True)

    valid = np.isfinite(pos).all(axis=1) & (norms[:, 0] > 0)

    out = np.full_like(pos, np.nan, dtype=float)
    out[valid] = pos[valid] / norms[valid]

    return out


def _standard_roi_target_positions(candidates):
    """
    Get approximate 10-20 / 10-05 target positions for ROI channel names.

    We use standard_1005 because it includes more intermediate channels
    such as CPz, CP1, and CP2.
    """
    montage = mne.channels.make_standard_montage("standard_1005")
    ch_pos = montage.get_positions()["ch_pos"]

    pos_by_upper = {
        str(name).upper(): np.asarray(pos, dtype=float)
        for name, pos in ch_pos.items()
    }

    target_positions = []
    target_names = []

    for ch in candidates:
        key = str(ch).upper()
        if key in pos_by_upper:
            target_positions.append(pos_by_upper[key])
            target_names.append(ch)

    # Conservative fallback if some candidate names are unavailable.
    if len(target_positions) == 0:
        for ch in ["Pz", "Cz"]:
            key = ch.upper()
            if key in pos_by_upper:
                target_positions.append(pos_by_upper[key])
                target_names.append(ch)

    if len(target_positions) == 0:
        return [], np.empty((0, 3), dtype=float)

    return target_names, np.vstack(target_positions)


def _nearest_montage_roi_indices(ch_names, candidates, sfreq=128.0, desired_k=None):
    """
    For non-standard channel names such as E1, E2, ..., choose channels whose
    montage positions are nearest to the requested centro-parietal ROI.

    This prevents ERP ROI analysis from falling back to channel 0 / E1.
    """
    info, montage_name, n_valid = make_eeg_info(ch_names, sfreq)
    picks = valid_topomap_picks(info)

    if len(picks) == 0:
        return [], None, None

    channel_positions = np.array(
        [info["chs"][idx]["loc"][:3] for idx in picks],
        dtype=float,
    )

    channel_positions_unit = _unit_normalize_positions(channel_positions)

    target_names, target_positions = _standard_roi_target_positions(candidates)

    if len(target_names) == 0:
        return [], montage_name, None

    target_positions_unit = _unit_normalize_positions(target_positions)

    selected = []

    # First, choose the nearest available EEG channel to each requested ROI target.
    for target_pos in target_positions_unit:
        if not np.isfinite(target_pos).all():
            continue

        distances = np.linalg.norm(channel_positions_unit - target_pos[None, :], axis=1)
        nearest_local = int(np.nanargmin(distances))
        nearest_global = int(picks[nearest_local])

        if nearest_global not in selected:
            selected.append(nearest_global)

    # Then fill with nearest channels to the ROI centroid, so the ROI is not only one channel.
    if desired_k is None:
        desired_k = min(len(candidates), len(picks))

    valid_target_positions = target_positions_unit[
        np.isfinite(target_positions_unit).all(axis=1)
    ]

    if len(valid_target_positions) > 0 and len(selected) < desired_k:
        centroid = valid_target_positions.mean(axis=0, keepdims=True)
        centroid = _unit_normalize_positions(centroid)[0]

        distances_to_centroid = np.linalg.norm(
            channel_positions_unit - centroid[None, :],
            axis=1,
        )

        sorted_local = np.argsort(distances_to_centroid)

        for local_idx in sorted_local:
            global_idx = int(picks[int(local_idx)])

            if global_idx not in selected:
                selected.append(global_idx)

            if len(selected) >= desired_k:
                break

    selected_names = [ch_names[i] for i in selected]

    return selected, montage_name, selected_names


def pick_roi_indices(ch_names, candidates):
    """
    Robust ROI selector for ERP sanity checks.

    Priority:
    1. Use exact 10-20/10-05 channel names if present: Pz, P3, P4, CPz, Cz, CP1, CP2.
    2. If exact names are unavailable, use montage-nearest channels.
    3. If no montage positions are available, fall back to Pz/Cz if present.
    4. As a last resort, use channel 0, but print a warning.
    """
    ch_names = [str(ch) for ch in ch_names]
    name_to_idx = {ch.upper(): i for i, ch in enumerate(ch_names)}

    # Exact standard ROI match.
    exact_idx = [
        name_to_idx[ch.upper()]
        for ch in candidates
        if ch.upper() in name_to_idx
    ]

    if len(exact_idx) > 0:
        selected_names = [ch_names[i] for i in exact_idx]
        print(f"[roi] exact ROI channels selected: {selected_names}")
        return exact_idx

    # Fallback exact Pz/Cz if present.
    fallback_idx = []
    for fb in ["PZ", "CZ"]:
        if fb in name_to_idx:
            fallback_idx.append(name_to_idx[fb])

    if len(fallback_idx) > 0:
        selected_names = [ch_names[i] for i in fallback_idx]
        print(f"[roi] fallback exact channels selected: {selected_names}")
        return fallback_idx

    # Montage-nearest ROI for EGI/HydroCel or other non-standard names.
    nearest_idx, montage_name, selected_names = _nearest_montage_roi_indices(
        ch_names,
        candidates,
        sfreq=128.0,
    )

    if len(nearest_idx) > 0:
        print(
            f"[roi] montage-nearest ROI selected using {montage_name}: "
            f"{selected_names}"
        )
        return nearest_idx

    # Last resort.
    print(
        "[WARN] Could not find exact or montage-nearest ROI channels. "
        f"Falling back to first channel: {ch_names[0]}"
    )
    return [0]

In [ ]:
all_summary_rows = []

for exp_name in EXPERIMENT_NAMES:
    print("=" * 90)
    print(exp_name)
    print("=" * 90)

    cfg = get_experiment(exp_name)

    data, summary_df, std_grand, tgt_grand, diff_grand = compute_erp_summary_for_dataset(cfg)

    all_summary_rows.append(summary_df)

    dataset_tag = f"{cfg.dataset.dataset}_{cfg.dataset.task}_{cfg.dataset.event_preset}_{cfg.run_tag}"

    times = data.times
    ch_names = data.ch_names
    sfreq = data.sfreq
    roi_idx = pick_roi_indices(ch_names, ROI_CANDIDATES)

    std_roi = std_grand[roi_idx].mean(axis=0)
    tgt_roi = tgt_grand[roi_idx].mean(axis=0)
    diff_roi = diff_grand[roi_idx].mean(axis=0)

    # ------------------------------------------------------------
    # Export clean ERP waveform data for final dissertation Figure 6.1
    # ------------------------------------------------------------

    for t, std_val, tgt_val, diff_val in zip(times, std_roi, tgt_roi, diff_roi):
        erp_waveform_rows.append({
            "experiment": exp_name,
            "dataset_label": cfg.dataset.dataset,
            "dataset": cfg.dataset.dataset,
            "task": cfg.dataset.task,
            "event_preset": cfg.dataset.event_preset,
            "run_tag": cfg.run_tag,
            "time_s": float(t),
            "standard_uv": float(std_val),
            "target_uv": float(tgt_val),
            "difference_uv": float(diff_val),
            "roi_channels": ", ".join([ch_names[i] for i in roi_idx]),
        })

    # ERP waveform
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.plot(times, std_roi, label="Standard")
    ax.plot(times, tgt_roi, label="Target")
    ax.axvline(0.0, linestyle="--", linewidth=1)
    ax.axvspan(P300_WINDOW[0], P300_WINDOW[1], alpha=0.15, label="P300 window")
    ax.set_title(f"ERP sanity check — {dataset_tag}")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Amplitude")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    save_figure(fig, FIG_DIR / f"{dataset_tag}_erp_standard_target.png")
    plt.show()

    # Difference waveform
    fig, ax = plt.subplots(figsize=(9, 4.2))
    ax.plot(times, diff_roi, label="Target − Standard")
    ax.axhline(0.0, linewidth=1)
    ax.axvline(0.0, linestyle="--", linewidth=1)
    ax.axvspan(P300_WINDOW[0], P300_WINDOW[1], alpha=0.15, label="P300 window")
    ax.set_title(f"Target-minus-standard ERP — {dataset_tag}")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Difference amplitude")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    save_figure(fig, FIG_DIR / f"{dataset_tag}_difference_waveform.png")
    plt.show()

    # Topography
    info, montage_name, n_valid = make_eeg_info(ch_names, sfreq)
    picks = valid_topomap_picks(info)

    p300_mask = (times >= P300_WINDOW[0]) & (times <= P300_WINDOW[1])

    if p300_mask.sum() == 0:
        print(f"[WARN] Skipping topography for {dataset_tag}: P300 window is empty.")

    else:
        topo_values = diff_grand[:, p300_mask].mean(axis=1)

        if len(picks) == 0:
            print(
                f"[WARN] Skipping scalp topomap for {dataset_tag}: "
                "no valid channel positions."
            )

            fallback_df = (
                pd.DataFrame({
                    "channel": ch_names,
                    "p300_diff_value": topo_values,
                    "abs_p300_diff_value": np.abs(topo_values),
                })
                .sort_values("abs_p300_diff_value", ascending=False)
                .reset_index(drop=True)
            )

            save_dataframe(
                fallback_df,
                TABLE_DIR / f"{dataset_tag}_p300_topography_fallback_channel_values.csv",
            )

            fig, ax = plt.subplots(figsize=(7, 5))
            top_df = fallback_df.head(15).iloc[::-1]

            ax.barh(top_df["channel"], top_df["p300_diff_value"])
            ax.axvline(0.0, linewidth=1)
            ax.set_title(
                f"P300-window channel differences\n"
                f"{dataset_tag}\n"
                f"no valid topomap montage"
            )
            ax.set_xlabel("Target − standard amplitude")
            ax.set_ylabel("Channel")
            ax.grid(True, axis="x", alpha=0.3)

            fig.tight_layout()
            save_figure(fig, FIG_DIR / f"{dataset_tag}_p300_top_channels_fallback.png")
            plt.show()

        else:
            info_valid = mne.pick_info(info, picks, copy=True)
            topo_values_valid = topo_values[picks]

            fig, ax = plt.subplots(figsize=(5, 4.5))

            mne.viz.plot_topomap(
                topo_values_valid,
                info_valid,
                axes=ax,
                show=False,
                contours=6,
            )

            ax.set_title(
                f"P300-window topography\n"
                f"{dataset_tag}\n"
                f"montage={montage_name}"
            )

            fig.tight_layout()
            save_figure(fig, FIG_DIR / f"{dataset_tag}_p300_topography.png")
            plt.show()

    save_dataframe(summary_df, TABLE_DIR / f"{dataset_tag}_erp_subject_summary.csv")

In [ ]:
erp_summary_all = pd.concat(all_summary_rows, ignore_index=True)
save_dataframe(erp_summary_all, TABLE_DIR / "erp_sanity_summary_all_datasets.csv")

erp_waveform_df = pd.DataFrame(erp_waveform_rows)
save_dataframe(
    erp_waveform_df,
    TABLE_DIR / "erp_difference_waveforms_long.csv",
)

display(round_numeric_columns(erp_summary_all, 4))
display(round_numeric_columns(erp_waveform_df.head(), 4))

print("Saved ERP summary:", TABLE_DIR / "erp_sanity_summary_all_datasets.csv")
print("Saved ERP waveform data:", TABLE_DIR / "erp_difference_waveforms_long.csv")

In [ ]:
erp_summary_all[[
    "dataset",
    "task",
    "event_preset",
    "subject",
    "roi_channels",
    "p300_mean_amp",
    "p300_peak_amp",
    "p300_peak_latency_s",
]].drop_duplicates().head(30)

In [ ]:
(
    erp_summary_all[["dataset", "event_preset", "roi_channels"]]
    .drop_duplicates()
    .sort_values(["dataset", "event_preset"])
)